# コエコミ — Colab バックエンド起動

このノートブックを **1台のColab = 1サーバー** として実行します。
本番は **6台**（Colab Pro+ 3アカウント × 2タブ）。`SERVER_ID` と `SERVER_COLOR` を台ごとに変えてください。

> **台数は「1台落ちても誰も気づかない」ためと、一斉に押されたときの待ち時間のためです。**
> 1台のGPUは同時に1行しか作らないので、その台に来た総行数 × 1行の秒数が
> そのまま最後の子の待ち時間になります。台数を倍にすると、その待ちが半分になります。
> 負荷分散はしていません（iPad 側が端末IDのハッシュで散らし、死んでいたら次の台に移ります）。

**⚠️ 同じアカウントで2台動かすときは、必ず別のノートブックを開いてください**
（例: アカウントA は `01` と `04`）。**同じファイルを2タブで開いてもランタイムは共有**され、
GPU は増えません。

**⚠️ ランタイムは GPU にしてください**（ランタイム > ランタイムのタイプを変更 > GPU）。
2つ目のセッションは GPU を取り逃すことがあります。取り逃すと**警告なしにCPUで動き**、
1行が数十秒かかってその台の全員が待たされます。起動ログ `[0/6]` の GPU 行を必ず確認してください。
GPU 無しで動作確認だけしたい場合は、セル2で `TTS_BACKEND='dummy'` にします。

**⚠️ イベントが終わったら必ずランタイムを停止してください。**
Pro+ のバックグラウンド実行はタブを閉じても動き続け、コンピューティングユニットを消費します。
6台ぶん動きっぱなしになると消費も6倍です。

| ノートブック | ID | color | label | 想定アカウント |
|----|----|-------|-------|------|
| `start_backend01.ipynb` | colab-1 | red | 赤サーバー | アカウントA |
| `start_backend02.ipynb` | colab-2 | blue | 青サーバー | アカウントB |
| `start_backend03.ipynb` | colab-3 | green | 緑サーバー | アカウントC |
| `start_backend04.ipynb` | colab-4 | yellow | 黄サーバー | アカウントA |
| `start_backend05.ipynb` | colab-5 | purple | 紫サーバー | アカウントB |
| `start_backend06.ipynb` | colab-6 | orange | オレンジサーバー | アカウントC |

In [ ]:
# 1) リポジトリを取得（2回目以降は最新に更新）
import os

if os.path.isdir("/content/koekomi/.git"):
    !cd /content/koekomi && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/TaiyoYamada/koekomi.git /content/koekomi
%cd /content/koekomi
!git log --oneline -1

In [ ]:
# 2) 設定（秘密情報は Colab の「シークレット」から読み込む。直書きしない）
import os
from google.colab import userdata  # 左の鍵アイコンから登録

# 名簿（今日のURLを配る先）
os.environ["GAS_URL"] = userdata.get("GAS_URL")

# イベントの合言葉。フロントの VITE_EVENT_TOKEN と同じ文字列にすること。
# 未設定だと誰でもこのAPIを叩けます（子どもの声を扱うので必ず設定）。
os.environ["EVENT_TOKEN"] = userdata.get("EVENT_TOKEN")

# 管理者トークン。後片付け（POST /cleanup = 全員分を消す）専用。
# 合言葉はフロントのバンドルに載る＝参加者なら誰でも読めるので、
# 「全消し」を守るには弱い。こちらはフロントに配らないこと。
# 未登録でも起動はできる（/cleanup が 503 になるだけ）ので、落とさず警告する。
try:
    os.environ["ADMIN_TOKEN"] = userdata.get("ADMIN_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("⚠️ ADMIN_TOKEN が未登録です。後片付け（/cleanup）は使えません。")

# フロントの公開オリジン。写真（4コマの絵）をサーバーが取りに行く先。
# 未設定だと動画をサーバーで作れず、iPad 側の書き出し（実時間）になります。
# 取得元は1つだけ。カスタムドメインを指定する。
os.environ["FRONTEND_ORIGIN"] = "https://koekomi.taiyoyamada.com"

# ブラウザから叩いてよいオリジン（CORS）。カンマ区切りで複数書ける。
# カスタムドメインと vercel.app の両方が生きているので、両方許可しておく。
# 片方だけにすると、もう片方を開いた子は全リクエストが弾かれて何もできない。
os.environ["CORS_ORIGINS"] = (
    "https://koekomi.taiyoyamada.com,"
    "https://koekomi.vercel.app"
)

# AIバックエンド: 既定で Qwen3-TTS（声クローン。GPU必須）。
# GPUが無い／動作確認だけなら次の行を有効にする:
# os.environ['TTS_BACKEND'] = 'dummy'

# この台の識別情報（台ごとに変える）
os.environ["SERVER_ID"] = "colab-4"
os.environ["SERVER_COLOR"] = "yellow"
os.environ["SERVER_LABEL"] = "黄サーバー"

# GPU 1枚なら 1 のまま。増やしても TTS_SERIALIZE=1 の間は GPU 並列度は 1。
os.environ["WORKERS"] = "1"

In [ ]:
# 3) 起動（依存インストール → FastAPI起動 → トンネル公開 → 名簿登録 → 自己チェック → heartbeat）
#    このセルは実行したままにしておく（Pro+ のバックグラウンド実行ならタブを閉じてもOK）。
%run colab/colab_runner.py

## 当日の確認

起動ログの最後に出る URL を使って、手元のPCから通しで確認します。

```bash
bash scripts/smoke-test.sh https://xxxx.trycloudflare.com <EVENT_TOKEN>
```

`/health` → `/voices` → `/jobs` → `/artifacts` → `/render` まで、
子どもがやるのと同じ順序で通します。**全部 PASS してから会場を開けてください。**
1行あたりの生成時間も表示されるので、待ち時間の見積もりにも使えます。